## Fine tuning

In [24]:
import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset as load_dataset_from_hub
from langchain_community.document_loaders import UnstructuredXMLLoader
import xml.etree.ElementTree as ET

# Configuration for environment
os.environ["WANDB_DISABLED"] = "true"

### Configuração geral

In [1]:
# Configuration
MODEL_ID = "MedicAmi/Meditron-7B"  # Foundation model para coisas de saúde/medicina
GITHUB_REPO_ID = "abachaa/MedQuAD" # Indicado pelo professor
DATA_FILE_PATH = "data.jsonl"         # Example filename in the repo

# Hiperparametros (Otimizado para rodar em uma RTX 3060 12GB)
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "cosine"
MAX_SAMPLES = 1000
MAX_SEQ_LENGTH = 512

### Carregando os dados do repositório git

In [11]:
import subprocess
from pathlib import Path

medquad_dir = Path("MedQuAD")

if not medquad_dir.is_dir():
    print("Pasta MedQuAD não encontrada. Clonando repositório...")
    subprocess.run(["git", "clone", "https://github.com/abachaa/MedQuAD.git"], check=True)
    print("Repositório clonado")
else:
    print("Pasta MedQuAD já existe, não é necessário clonar novamente.")

Pasta MedQuAD não encontrada. Clonando repositório...
Repositório clonado


In [34]:
def parse_file(file_path):
    tree = ET.parse(file_path)
    qapairs = tree.iter("QAPair")

    parsed_data = []

    for qapair in qapairs:
        question_elem = qapair.find("Question")
        answer_elem = qapair.find("Answer")

        if question_elem is not None and answer_elem is not None:
            question = question_elem.text
            answer = answer_elem.text

            if question is not None and answer is not None:
                parsed_data.append({"question": question, "answer": answer})
    return parsed_data

In [36]:
docs = []

for folder in os.listdir(medquad_dir):
    folder_path = medquad_dir / folder
    if folder_path.is_dir():
        for file in os.listdir(folder_path):
            if file.endswith(".xml"):
                file_path = folder_path / file
                docs.extend(parse_file(file_path))

df = pd.DataFrame(docs)

In [37]:
df

,question,answer
0,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points\n - Adult acute ...
1,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...
...,...,...
16402,What is (are) Parasites - Zoonotic Hookworm ?,"There are many different species of hookworms,..."
16403,Who is at risk for Parasites - Zoonotic Hookwo...,Dog and cat hookworms are found throughout the...
16404,How to diagnose Parasites - Zoonotic Hookworm ?,Cutaneous larva migrans (CLM) is a clinical di...
16405,What are the treatments for Parasites - Zoonot...,The zoonotic hookworm larvae that cause cutane...


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    # Adjust 'instruction' and 'output' keys based on your dataset structure
    inputs = [f"Instruction: {i}\nResponse: {o}" for i, o in zip(examples['instruction'], examples['output'])]
    model_inputs = tokenizer(inputs, max_length=MAX_SEQ_LENGTH, truncation=True, padding="max_length")
    
    # Labels are the same as input_ids for causal LM
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
# Config for 4-bit quantization to fit on RTX 3060
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Standard for many models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./meditron-finetuned",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=3,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    logging_steps=10,
    fp16=True, # Required for RTX 3060
    save_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True),
)

trainer.train()